In [3]:
from pathlib import Path
import math
import random
from typing import Optional

import cv2
import numpy as np
import pandas as pd
from PIL import Image
import openslide
import pydicom
from tqdm import tqdm
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_PATH = Path("../../data")
PROJECT_DATA_PATH = DATA_PATH / "pancreatic_cancer_pathology"
RAW_PATH = PROJECT_DATA_PATH / "raw"
DST_PATH = PROJECT_DATA_PATH / "dst"
IMAGE_DST_PATH = DST_PATH / "Image"
TCGA_RAW_PATH = RAW_PATH / "TCGA_PAAD"
CPTAC_RAW_PATH = RAW_PATH / "CPTAC_PDAC"

TARGET_MPP = 1.0
TILE_SIZE = 1024
TISSUE_THRESHOLD = 0.15
SKIP_EXISTING = True
OVERWRITE_METADATA = False
DEBUG_MAX_CASES: Optional[int] = None  # 빠른 테스트 시 1 또는 3으로 변경

IMAGE_DST_PATH.mkdir(parents=True, exist_ok=True)

print("PROJECT_DATA_PATH:", PROJECT_DATA_PATH.resolve())
print("IMAGE_DST_PATH:", IMAGE_DST_PATH.resolve())
print("TARGET_MPP:", TARGET_MPP, "TILE_SIZE:", TILE_SIZE)


def safe_name(value: object) -> str:
    return str(value).replace("/", "_").replace(" ", "_")


def tissue_fraction(rgb: np.ndarray) -> float:
    if rgb.size == 0:
        return 0.0
    rgb = rgb[..., :3].astype(np.uint8)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    saturation = hsv[..., 1]
    value = hsv[..., 2]
    tissue_mask = (saturation > 20) & (value < 245)
    return float(tissue_mask.mean())


def save_tile(tile: np.ndarray, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(tile.astype(np.uint8)).save(out_path)


def get_openslide_mpp(slide: openslide.OpenSlide) -> float:
    props = slide.properties
    for key in [openslide.PROPERTY_NAME_MPP_X, "aperio.MPP", "openslide.mpp-x"]:
        value = props.get(key)
        if value is not None:
            return float(value)
    objective = props.get(openslide.PROPERTY_NAME_OBJECTIVE_POWER)
    if objective is not None:
        objective = float(objective)
        if objective > 0:
            return 10.0 / objective
    raise ValueError("WSI mpp 정보를 찾지 못했습니다. slide.properties를 확인하세요.")


def get_dicom_mpp(ds: pydicom.Dataset) -> float:
    if hasattr(ds, "SharedFunctionalGroupsSequence"):
        shared = ds.SharedFunctionalGroupsSequence[0]
        if hasattr(shared, "PixelMeasuresSequence"):
            spacing = shared.PixelMeasuresSequence[0].PixelSpacing
            return float(spacing[0]) * 1000.0  # mm/pixel -> um/pixel
    for key in ["PixelSpacing", "ImagerPixelSpacing", "NominalScannedPixelSpacing"]:
        spacing = getattr(ds, key, None)
        if spacing is not None:
            return float(spacing[0]) * 1000.0
    raise ValueError("DICOM mpp 정보를 찾지 못했습니다.")


def normalize_dicom_frames(pixel_array: np.ndarray) -> np.ndarray:
    if pixel_array.ndim == 4:
        return pixel_array[..., :3]
    if pixel_array.ndim == 3 and pixel_array.shape[-1] in (3, 4):
        return pixel_array[None, ..., :3]
    if pixel_array.ndim == 3:
        return np.repeat(pixel_array[..., None], 3, axis=-1)
    if pixel_array.ndim == 2:
        return np.repeat(pixel_array[None, ..., None], 3, axis=-1)
    raise ValueError(f"예상하지 못한 DICOM pixel_array shape입니다: {pixel_array.shape}")


def get_dicom_region(
    frames: np.ndarray,
    x: int,
    y: int,
    source_size: int,
    total_cols: int,
    total_rows: int,
) -> np.ndarray:
    tile_rows, tile_cols = frames.shape[1], frames.shape[2]
    grid_cols = math.ceil(total_cols / tile_cols)
    grid_rows = math.ceil(total_rows / tile_rows)
    region = np.full((source_size, source_size, 3), fill_value=255, dtype=frames.dtype)

    start_col = x // tile_cols
    end_col = min(grid_cols - 1, (x + source_size - 1) // tile_cols)
    start_row = y // tile_rows
    end_row = min(grid_rows - 1, (y + source_size - 1) // tile_rows)

    for row_idx in range(start_row, end_row + 1):
        for col_idx in range(start_col, end_col + 1):
            frame_idx = row_idx * grid_cols + col_idx
            if frame_idx >= frames.shape[0]:
                continue
            frame = frames[frame_idx]
            frame_x0 = col_idx * tile_cols
            frame_y0 = row_idx * tile_rows
            overlap_x0 = max(x, frame_x0)
            overlap_y0 = max(y, frame_y0)
            overlap_x1 = min(x + source_size, frame_x0 + tile_cols, total_cols)
            overlap_y1 = min(y + source_size, frame_y0 + tile_rows, total_rows)
            if overlap_x1 <= overlap_x0 or overlap_y1 <= overlap_y0:
                continue
            src_x0 = overlap_x0 - frame_x0
            src_y0 = overlap_y0 - frame_y0
            dst_x0 = overlap_x0 - x
            dst_y0 = overlap_y0 - y
            width = overlap_x1 - overlap_x0
            height = overlap_y1 - overlap_y0
            region[dst_y0:dst_y0 + height, dst_x0:dst_x0 + width] = frame[
                src_y0:src_y0 + height, src_x0:src_x0 + width
            ]
    return region


def select_cptac_volume_dicom(series_dir: Path, target_mpp: float) -> tuple[Path, pd.DataFrame]:
    records = []
    for path in sorted(series_dir.glob("*.dcm"), key=lambda p: p.stat().st_size):
        ds_meta = pydicom.dcmread(path, stop_before_pixels=True)
        image_type = " | ".join(map(str, getattr(ds_meta, "ImageType", []))).upper()
        if "VOLUME" not in image_type:
            continue
        mpp = get_dicom_mpp(ds_meta)
        records.append(
            {
                "path": path,
                "file_name": path.name,
                "image_type": image_type,
                "mpp": mpp,
                "rows": int(ds_meta.Rows),
                "columns": int(ds_meta.Columns),
                "number_of_frames": int(getattr(ds_meta, "NumberOfFrames", 1)),
                "total_rows": int(getattr(ds_meta, "TotalPixelMatrixRows", ds_meta.Rows)),
                "total_cols": int(getattr(ds_meta, "TotalPixelMatrixColumns", ds_meta.Columns)),
                "size_mb": path.stat().st_size / (1024 ** 2),
            }
        )
    meta_df = pd.DataFrame(records)
    if meta_df.empty:
        raise ValueError(f"VOLUME DICOM을 찾지 못했습니다: {series_dir}")
    finer_or_equal = meta_df[meta_df["mpp"] <= target_mpp].copy()
    if finer_or_equal.empty:
        selected = meta_df.iloc[(meta_df["mpp"] - target_mpp).abs().argsort()].iloc[0]
    else:
        selected = finer_or_equal.sort_values("mpp", ascending=False).iloc[0]
    return Path(selected["path"]), meta_df


def resize_to_target(tile: np.ndarray) -> np.ndarray:
    return cv2.resize(tile[..., :3], (TILE_SIZE, TILE_SIZE), interpolation=cv2.INTER_AREA)


PROJECT_DATA_PATH: /home/user/urbandatalab/YSLee/data/pancreatic_cancer_pathology
IMAGE_DST_PATH: /home/user/urbandatalab/YSLee/data/pancreatic_cancer_pathology/dst/Image
TARGET_MPP: 1.0 TILE_SIZE: 1024


In [ ]:
# TCGA-PAAD SVS WSI tiling: mpp 1.0, 1024 x 1024

TCGA_DATASET_NAME = "TCGA_PAAD"
TCGA_COHORT_PATH = Path("outputs/data_verification/tcga_survival_cohort_candidate.csv")
TCGA_IMAGE_DST = IMAGE_DST_PATH / TCGA_DATASET_NAME
TCGA_IMAGE_DST.mkdir(parents=True, exist_ok=True)

tcga_df = pd.read_csv(TCGA_COHORT_PATH)
tcga_df = tcga_df[tcga_df["wsi_exists"].astype(bool)].copy()
if DEBUG_MAX_CASES is not None:
    tcga_df = tcga_df.head(DEBUG_MAX_CASES).copy()

tcga_summary = []

for _, row in tcga_df.iterrows():
    case_id = safe_name(row["patient_id"])
    wsi_path = Path(row["wsi_path"])
    case_dir = TCGA_IMAGE_DST / case_id
    case_dir.mkdir(parents=True, exist_ok=True)
    metadata_path = case_dir / "tile_metadata.csv"

    if SKIP_EXISTING and metadata_path.exists() and not OVERWRITE_METADATA:
        print(f"[SKIP] {case_id}: existing metadata")
        continue

    slide = openslide.OpenSlide(str(wsi_path))
    native_mpp = get_openslide_mpp(slide)
    source_tile_size = int(round(TILE_SIZE * TARGET_MPP / native_mpp))
    width, height = slide.dimensions

    tile_records = []
    tile_idx = 0
    n_candidate = 0
    for y in tqdm(range(0, height - source_tile_size + 1, source_tile_size), desc=f"Processing {case_id} rows"):
        for x in range(0, width - source_tile_size + 1, source_tile_size):
            n_candidate += 1
            out_path = case_dir / f"{case_id}_x{x}_y{y}_mpp{TARGET_MPP:.1f}_{tile_idx:06d}.png"
            if SKIP_EXISTING and out_path.exists():
                tile_idx += 1
                continue
            region = slide.read_region((x, y), 0, (source_tile_size, source_tile_size)).convert("RGB")
            tile = resize_to_target(np.asarray(region))
            frac = tissue_fraction(tile)
            if frac < TISSUE_THRESHOLD:
                continue
            save_tile(tile, out_path)
            tile_records.append(
                {
                    "dataset": TCGA_DATASET_NAME,
                    "case_id": case_id,
                    "tile_path": out_path.as_posix(),
                    "wsi_path": wsi_path.as_posix(),
                    "x_level0": x,
                    "y_level0": y,
                    "source_tile_size_px": source_tile_size,
                    "target_tile_size_px": TILE_SIZE,
                    "native_mpp": native_mpp,
                    "target_mpp": TARGET_MPP,
                    "tissue_fraction": frac,
                    "OS_time": row.get("OS_time"),
                    "OS_event": row.get("OS_event"),
                }
            )
            tile_idx += 1
    slide.close()

    tile_meta_df = pd.DataFrame(tile_records)
    tile_meta_df.to_csv(metadata_path, index=False)
    tcga_summary.append(
        {
            "dataset": TCGA_DATASET_NAME,
            "case_id": case_id,
            "wsi_path": wsi_path.as_posix(),
            "native_mpp": native_mpp,
            "source_tile_size_px": source_tile_size,
            "n_candidate_tiles": n_candidate,
            "n_saved_tiles": len(tile_meta_df),
            "case_dir": case_dir.as_posix(),
            "metadata_path": metadata_path.as_posix(),
        }
    )
    print(f"[TCGA] {case_id}: saved {len(tile_meta_df)} / {n_candidate} tiles")

tcga_summary_df = pd.DataFrame(tcga_summary)
tcga_summary_path = TCGA_IMAGE_DST / "tile_summary.csv"
tcga_summary_df.to_csv(tcga_summary_path, index=False)
display(tcga_summary_df.head())
print("saved:", tcga_summary_path)


Processing TCGA-2J-AAB1 rows:  18%|█▊        | 3/17 [01:13<06:20, 27.17s/it]

In [ ]:
# CPTAC-PDAC DICOM WSI tiling: mpp 1.0, 1024 x 1024
# CPTAC DICOM은 VOLUME frame tile을 row-major grid로 보고 원본 VOLUME에서 mpp 1.0 tile을 생성합니다.

CPTAC_DATASET_NAME = "CPTAC_PDAC"
CPTAC_COHORT_PATH = Path("outputs/data_verification/cptac_survival_cohort_candidate.csv")
CPTAC_WSI_SERIES_PATH = CPTAC_RAW_PATH / "matched" / "cptac_pda_matched_wsi_series.csv"
CPTAC_IMAGE_DST = IMAGE_DST_PATH / CPTAC_DATASET_NAME
CPTAC_IMAGE_DST.mkdir(parents=True, exist_ok=True)

cptac_df = pd.read_csv(CPTAC_COHORT_PATH)
cptac_wsi_df = pd.read_csv(CPTAC_WSI_SERIES_PATH)
cptac_df = cptac_df[cptac_df["has_wsi_series"].astype(bool)].copy()
if DEBUG_MAX_CASES is not None:
    cptac_df = cptac_df.head(DEBUG_MAX_CASES).copy()

cptac_summary = []

for _, row in cptac_df.iterrows():
    case_id = safe_name(row["case_id"])
    case_dir = CPTAC_IMAGE_DST / case_id
    case_dir.mkdir(parents=True, exist_ok=True)
    metadata_path = case_dir / "tile_metadata.csv"

    if SKIP_EXISTING and metadata_path.exists() and not OVERWRITE_METADATA:
        print(f"[SKIP] {case_id}: existing metadata")
        continue

    case_series = cptac_wsi_df[cptac_wsi_df["case_id"].eq(row["case_id"])].copy()
    tumor_series = case_series[
        case_series["SeriesDescription"].astype(str).str.contains("tumor", case=False, na=False)
    ]
    if tumor_series.empty:
        print(f"[SKIP] {case_id}: HE tumor series 없음")
        continue
    selected_series = tumor_series.sort_values("series_size_MB", ascending=False).iloc[0]
    series_dirs = list((CPTAC_RAW_PATH / "WSI_DICOM").rglob(f"SM_{selected_series['SeriesInstanceUID']}"))
    if len(series_dirs) == 0:
        print(f"[SKIP] {case_id}: DICOM series directory 없음")
        continue
    series_dir = series_dirs[0]
    volume_path, volume_meta_df = select_cptac_volume_dicom(series_dir, TARGET_MPP)
    volume_meta_df.to_csv(case_dir / "dicom_volume_candidates.csv", index=False)

    ds = pydicom.dcmread(volume_path)
    native_mpp = get_dicom_mpp(ds)
    frames = normalize_dicom_frames(ds.pixel_array)
    total_rows = int(getattr(ds, "TotalPixelMatrixRows", frames.shape[1]))
    total_cols = int(getattr(ds, "TotalPixelMatrixColumns", frames.shape[2]))
    source_tile_size = int(round(TILE_SIZE * TARGET_MPP / native_mpp))

    tile_records = []
    tile_idx = 0
    n_candidate = 0
    for y in tqdm(range(0, total_rows - source_tile_size + 1, source_tile_size), desc=f"Processing {case_id} rows"):
        for x in range(0, total_cols - source_tile_size + 1, source_tile_size):
            n_candidate += 1
            out_path = case_dir / f"{case_id}_x{x}_y{y}_mpp{TARGET_MPP:.1f}_{tile_idx:06d}.png"
            if SKIP_EXISTING and out_path.exists():
                tile_idx += 1
                continue
            source_region = get_dicom_region(frames, x, y, source_tile_size, total_cols, total_rows)
            tile = resize_to_target(source_region)
            frac = tissue_fraction(tile)
            if frac < TISSUE_THRESHOLD:
                continue
            save_tile(tile, out_path)
            tile_records.append(
                {
                    "dataset": CPTAC_DATASET_NAME,
                    "case_id": case_id,
                    "tile_path": out_path.as_posix(),
                    "volume_dicom_path": volume_path.as_posix(),
                    "SeriesInstanceUID": selected_series["SeriesInstanceUID"],
                    "SeriesDescription": selected_series["SeriesDescription"],
                    "x_total_matrix": x,
                    "y_total_matrix": y,
                    "source_tile_size_px": source_tile_size,
                    "target_tile_size_px": TILE_SIZE,
                    "native_mpp": native_mpp,
                    "target_mpp": TARGET_MPP,
                    "tissue_fraction": frac,
                    "OS_time": row.get("OS_time"),
                    "OS_event": row.get("OS_event"),
                }
            )
            tile_idx += 1

    tile_meta_df = pd.DataFrame(tile_records)
    tile_meta_df.to_csv(metadata_path, index=False)
    cptac_summary.append(
        {
            "dataset": CPTAC_DATASET_NAME,
            "case_id": case_id,
            "SeriesInstanceUID": selected_series["SeriesInstanceUID"],
            "SeriesDescription": selected_series["SeriesDescription"],
            "volume_dicom_path": volume_path.as_posix(),
            "native_mpp": native_mpp,
            "source_tile_size_px": source_tile_size,
            "n_candidate_tiles": n_candidate,
            "n_saved_tiles": len(tile_meta_df),
            "case_dir": case_dir.as_posix(),
            "metadata_path": metadata_path.as_posix(),
        }
    )
    print(f"[CPTAC] {case_id}: saved {len(tile_meta_df)} / {n_candidate} tiles")

cptac_summary_df = pd.DataFrame(cptac_summary)
cptac_summary_path = CPTAC_IMAGE_DST / "tile_summary.csv"
cptac_summary_df.to_csv(cptac_summary_path, index=False)
display(cptac_summary_df.head())
print("saved:", cptac_summary_path)
